In [1]:
import warnings
warnings.filterwarnings("ignore")

from rfdetr import RFDETRLarge
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import os

In [ ]:
ds_dir = Path(f"runs/rfdetr_large/run{len(os.listdir(r'runs/rfdetr_large')) + 1}")
image_dir = Path(r"data/raw/test/test")

In [ ]:
NUM_CLASSES = 8

model = RFDETRLarge(
    pretrain_weights= ds_dir / "checkpoint_best_ema.pth"
)

rows = []
row_id = 0

for image_path in tqdm(list(image_dir.glob("*")), desc="Predicting"):

    detections = model.predict(
        str(image_path),
        threshold=0.01
    )

    for box, conf, cls_id in zip(
        detections.xyxy,
        detections.confidence,
        detections.class_id
    ):
        cls_id = int(cls_id)

        # Skip background / invalid class
        if not 0 <= cls_id < NUM_CLASSES:
            continue

        x1, y1, x2, y2 = map(float, box)

        # Skip degenerate boxes
        if x2 <= x1 or y2 <= y1:
            continue

        rows.append({
            "id": row_id,
            "image_id": image_path.name,
            "class_id": cls_id,
            "confidence": float(conf),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

        row_id += 1

df_pred = pd.DataFrame(
    rows,
    columns=[
        "id",
        "image_id",
        "class_id",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
    ]
)

# Final validation
assert df_pred["class_id"].between(0, 7).all()
assert (df_pred["x2"] > df_pred["x1"]).all()
assert (df_pred["y2"] > df_pred["y1"]).all()

df_pred.to_csv(
    ds_dir / "submission.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Predictions:", len(df_pred))

[2026-09-29 00:39:06] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-29 00:39:06] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-29 00:39:06] [WARNING] rf-detr - Checkpoint has 8 classes but model is configured for 90. Using checkpoint class count (8). Pass num_classes=8 to suppress this warning.
Predicting: 100%|██████████| 1013/1013 [00:41<00:00, 24.48it/s]


Predictions: 272962
